# PWV02 : Seasonal Variation — Gaussian Process fit — Publication-quality figures

**Goal :** Demonstrate annual PWV variations at Cerro Pachon with
- **Gaussian Process regression** (full timeline 2022-2026, RBF + periodic kernel)
- **Year-folded** (phase) plot Jan→Dec with one colour per year
- GP mean curve + **±2σ confidence band** (shaded fill) replacing the monthly mean diamond markers
- Error bars on PWV, markers/colours by filter or by year
- Compact 2-panel layout with GridSpec (full timeline + seasonal phase)
- Seasonal statistics table (Summer/Autumn/Winter/Spring) + LaTeX export

**Author :** Sylvie Dagoret-Campagne — IJCLab/IN2P3/CNRS  
**Created :** 2026-03-19  
**Kernel :** conda_py313

---
### GP kernel strategy

| Panel | Kernel | Purpose |
|-------|--------|---------|
| Full timeline | `RBF(l≈180 d) × ExpSineSquared(p=365.25 d)` + `WhiteKernel` | long-range trend + annual periodicity + noise |
| Folded (phase) | `RBF(l≈60 d) + WhiteKernel` on doy_norm ∈[0,1] | smooth seasonal shape |

Both GPs are trained on a **daily-median** grid to keep the fit tractable
while preserving the seasonal signal.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.resetwarnings()
warnings.simplefilter('ignore')

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from astropy.time import Time

# ── scikit-learn Gaussian Process ─────────────────────────────────────────────
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, ExpSineSquared, WhiteKernel, ConstantKernel as C
)

%matplotlib inline

# ── publication-style rc params ───────────────────────────────────────────────
mpl.rcParams.update({
    'figure.dpi'     : 150,
    'font.family'    : 'serif',
    'font.size'      : 11,
    'axes.labelsize' : 13,
    'axes.titlesize' : 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.grid'      : True,
    'grid.alpha'     : 0.35,
    'axes.linewidth' : 1.1,
})

In [ ]:
from PWV00_parameters import (
    version_run, tag, extractedfilesdict, legendtag,
    PWV_FILTER_LIST, FLAG_PWVFILTERS,
    DumpConfig
)
from mysitcom.auxtel.qualitycuts import ParameterCutSelection, ParameterCutTools
from mysitcom.auxtel.pwv import normalize_column_data_bytarget_byfilter

DumpConfig()

In [ ]:
# ── output directories ────────────────────────────────────────────────────────
pathfigs = "figs_PWV02gp"
prefix   = "pwv02gp"
os.makedirs(pathfigs, exist_ok=True)
figtype  = ".pdf"   # vector for journal submission; change to .png for quick preview

In [ ]:
# ── PWV hard cuts ─────────────────────────────────────────────────────────────
PWVMIN = 0.
PWVMAX = 16.

---
## 1. Load data

In [ ]:
# hack pour la lecture des pickes
import numpy as np
import types
import sys
sys.modules['numpy.rec'] = np.rec

In [ ]:
atmfilename   = extractedfilesdict[version_run]
inputfilename = atmfilename.split("/")[-1]
print(f"Loading: {atmfilename}")

if "parquet" in inputfilename:
    df_spec = pd.read_parquet(atmfilename)
else:
    specdata = np.load(atmfilename, allow_pickle=True)
    df_spec  = pd.DataFrame(specdata)

# ── Build Time column from DATE-OBS ───────────────────────────────────────────
df_spec["Time"] = pd.to_datetime(df_spec["DATE-OBS"], utc=True)
df_spec = df_spec.sort_values(by="Time", ascending=True).reset_index(drop=True)

print(f"Shape: {df_spec.shape}")
print(f"Time range: {df_spec['Time'].min()}  →  {df_spec['Time'].max()}")
print("Columns:", ", ".join(df_spec.columns))

In [ ]:
# ── rename Corentin run columns to standardised names ─────────────────────────
if "run2026_v02" in version_run:
    df_spec["chi2_ram"] = df_spec["CHI2_FIT"]
    df_spec["chi2_rum"] = df_spec["CHI2_FIT"]

# ── keep only PWV-sensitive filters ───────────────────────────────────────────
if FLAG_PWVFILTERS:
    df_spec = df_spec[df_spec["FILTER"].isin(PWV_FILTER_LIST)].copy()

# ── drop rows with missing PWV ────────────────────────────────────────────────
pwv_cols = ["PWV [mm]_ram", "PWV [mm]_rum", "PWV [mm]_err_ram", "PWV [mm]_err_rum"]
df_spec.dropna(subset=pwv_cols, inplace=True)

print(f"After filter selection: {len(df_spec)} rows")
print("Filters present:", df_spec["FILTER"].unique())

In [ ]:
# ── normalise chi2 per target per filter ──────────────────────────────────────
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="CHI2_FIT", ext="norm")
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="chi2_ram", ext="norm")
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="chi2_rum", ext="norm")

---
## 2. Apply quality cuts

In [ ]:
FLAG_LOOSE_CUTS = True
FLAG_TIGHT_CUTS = False

In [ ]:
pathdata      = "data_PWV01seas"   # reuse cuts from PWV01 notebook
if FLAG_LOOSE_CUTS:
    filename_cuts = f"{pathdata}/cuts_loose_finaldecision.json" 
    cutstype_name = "loose-cuts"
elif FLAG_TIGHT_CUTS: 
    filename_cuts = f"{pathdata}/cuts_tight_finaldecision.json" 
    cutstype_name = "tight-cuts"
else:
    filename_cuts = f"{pathdata}/cuts_finaldecision.json" 
    cutstype_name = "standard-cuts"

cuts           = ParameterCutTools.load_cuts_json(filename_cuts)
list_of_params = list(cuts.keys())

selector    = ParameterCutSelection(df_spec, params=list_of_params, id_col="id")
flags       = selector.apply_cuts(cuts)
df_selected = df_spec.merge(flags, on="id")
df_keep     = df_selected[df_selected["pass_all_cuts"]].copy()

print(f"Quality cuts: {len(df_keep)} / {len(df_spec)} observations kept")

---
## 3. Derived time columns

In [ ]:
# ── constants ─────────────────────────────────────────────────────────────────
MJD_J2000 = 51544.5   # 2000-01-01T12:00 UTC
YEAR_DAYS  = 365.25

MONTH_ABBR = ["Jan","Feb","Mar","Apr","May","Jun",
              "Jul","Aug","Sep","Oct","Nov","Dec"]

# ── calendar fields ───────────────────────────────────────────────────────────
df_keep["year"]     = df_keep["Time"].dt.year
df_keep["month"]    = df_keep["Time"].dt.month
df_keep["doy"]      = df_keep["Time"].dt.day_of_year
df_keep["doy_norm"] = (df_keep["doy"] - 1) / 365.25    # 0→1  Jan→Dec
df_keep["mjd_year"] = (df_keep["MJD"] - MJD_J2000) / YEAR_DAYS

mjd_min = df_keep["MJD"].min()
mjd_max = df_keep["MJD"].max()

# ── best-estimate PWV and combined uncertainty ────────────────────────────────
df_keep["PWV"]     = 0.5 * (df_keep["PWV [mm]_ram"] + df_keep["PWV [mm]_rum"])
df_keep["PWV_err"] = np.sqrt(
    df_keep["PWV [mm]_err_ram"]**2 + df_keep["PWV [mm]_err_rum"]**2
)

years_present = sorted(df_keep["year"].dropna().astype(int).unique())
print("Years in dataset :", years_present)
print(f"MJD range        : {mjd_min:.1f} — {mjd_max:.1f}")
print(f"Date range       : {df_keep['Time'].min().date()}  →  {df_keep['Time'].max().date()}")

---
## 4. Build daily-median training set for GPs

The raw dataset contains many observations per day.
We build a **daily-weighted-median** representation to speed up GP fitting
while retaining the seasonal signal.
The daily noise level is estimated from the intra-day spread.

In [ ]:
# ── daily grouping ────────────────────────────────────────────────────────────
df_keep["date"] = df_keep["Time"].dt.date

def daily_stats(g):
    w   = 1.0 / (g["PWV_err"].values**2 + 1e-9)
    mu  = np.average(g["PWV"].values, weights=w)
    # combined uncertainty: quadrature of measurement uncertainty + intra-day scatter
    sigma_meas = 1.0 / np.sqrt(w.sum())
    sigma_scat = g["PWV"].std(ddof=0) if len(g) > 1 else 0.0
    sigma_day  = np.sqrt(sigma_meas**2 + sigma_scat**2)
    return pd.Series({
        "MJD"     : g["MJD"].mean(),
        "doy_norm": g["doy_norm"].mean(),
        "PWV"     : mu,
        "PWV_err" : sigma_day,
        "n"       : len(g),
    })

df_daily = (
    df_keep
    .groupby("date", group_keys=False)
    .apply(daily_stats)
    .reset_index(drop=True)
    .sort_values("MJD")
    .reset_index(drop=True)
)

print(f"Daily points for GP fit: {len(df_daily)}")
print(df_daily.describe())

---
## 5. Gaussian Process — full timeline (2022-2026)

**Kernel :**  
$$k(t_1,t_2) = C \times \text{RBF}(l_{\rm slow}) \times \text{ExpSineSquared}(l_{\rm ESS}, p=1\,{\rm yr}) + \text{WhiteKernel}$$

where $t$ is in **days** (MJD).  
The `ExpSineSquared` kernel enforces the annual periodicity;
the outer `RBF` allows the amplitude to drift slowly over years;
the `WhiteKernel` absorbs residual observational noise.

In [ ]:
# ── Training data (MJD in days) ───────────────────────────────────────────────
X_train_full = df_daily["MJD"].values.reshape(-1, 1)
y_train_full = df_daily["PWV"].values
alpha_full   = df_daily["PWV_err"].values**2   # noise variance per point

# ── Kernel definition ─────────────────────────────────────────────────────────
#  Slow amplitude modulation (length scale ~ 2 years = 730 d)
#  × Annual periodic component (period = 365.25 d, length scale ~ 60 d)
#  + White noise
k_trend    = C(1.0, (0.1, 10.0)) * RBF(length_scale=730, length_scale_bounds=(200, 2000))
k_periodic = ExpSineSquared(
    length_scale=60,
    periodicity=365.25,
    length_scale_bounds=(20, 400),
    periodicity_bounds=(300, 400),
)
k_noise    = WhiteKernel(noise_level=1.0, noise_level_bounds=(0.01, 10.0))

kernel_full = k_trend * k_periodic + k_noise

gp_full = GaussianProcessRegressor(
    kernel=kernel_full,
    alpha=alpha_full,
    n_restarts_optimizer=5,
    normalize_y=True,
)

print("Fitting GP on full timeline …")
gp_full.fit(X_train_full, y_train_full)
print("Done.")
print(f"Optimised kernel: {gp_full.kernel_}")
print(f"Log-marginal-likelihood: {gp_full.log_marginal_likelihood_value_:.2f}")

In [ ]:
# ── Prediction grid (full timeline, ~1-day steps) ─────────────────────────────
mjd_pred_full  = np.linspace(mjd_min - 20, mjd_max + 20, 1500).reshape(-1, 1)
gp_mean_full, gp_std_full = gp_full.predict(mjd_pred_full, return_std=True)

# Clip negative PWV to 0 in the mean curve
gp_mean_full = np.clip(gp_mean_full, 0, None)

print(f"GP mean range  : [{gp_mean_full.min():.2f}, {gp_mean_full.max():.2f}] mm")
print(f"GP 2σ max band : {2*gp_std_full.max():.2f} mm")

---
## 6. Gaussian Process — year-folded (seasonal phase)

**Kernel :**  
$$k(\phi_1,\phi_2) = C \times \text{RBF}(l_{\rm phase}) + \text{WhiteKernel}$$

where $\phi \in [0,1]$ is the fractional day-of-year.
The training set is the daily-median data folded to $[0,1]$;
we replicate it three times (−1, 0, +1) to enforce periodicity at the boundaries.

In [ ]:
# ── Training data (phase ∈ [0,1]) — replicated for periodicity ───────────────
phi_raw   = df_daily["doy_norm"].values
pwv_raw   = df_daily["PWV"].values
err_raw   = df_daily["PWV_err"].values

# Tile: [-1,0], [0,1], [1,2]  → GP sees a periodic signal
phi_tile  = np.concatenate([phi_raw - 1, phi_raw, phi_raw + 1])
pwv_tile  = np.tile(pwv_raw, 3)
err_tile  = np.tile(err_raw, 3)

X_train_phase = phi_tile.reshape(-1, 1)
y_train_phase = pwv_tile
alpha_phase   = err_tile**2

# ── Kernel: smooth periodic RBF ───────────────────────────────────────────────
# length scale ~ 0.10 in fractional year ≈ 36 d
k_phase = C(1.0, (0.1, 10.0)) * RBF(length_scale=0.10, length_scale_bounds=(0.03, 0.5))
k_phase_noise = WhiteKernel(noise_level=0.5, noise_level_bounds=(0.01, 5.0))

kernel_phase = k_phase + k_phase_noise

gp_phase = GaussianProcessRegressor(
    kernel=kernel_phase,
    alpha=alpha_phase,
    n_restarts_optimizer=5,
    normalize_y=True,
)

print("Fitting GP on seasonal phase …")
gp_phase.fit(X_train_phase, y_train_phase)
print("Done.")
print(f"Optimised kernel: {gp_phase.kernel_}")
print(f"Log-marginal-likelihood: {gp_phase.log_marginal_likelihood_value_:.2f}")

In [ ]:
# ── Prediction grid (phase ∈ [0,1]) ──────────────────────────────────────────
phi_pred = np.linspace(0, 1, 500).reshape(-1, 1)
gp_mean_phase, gp_std_phase = gp_phase.predict(phi_pred, return_std=True)
gp_mean_phase = np.clip(gp_mean_phase, 0, None)

phi_pred_flat = phi_pred.ravel()

print(f"GP phase mean range : [{gp_mean_phase.min():.2f}, {gp_mean_phase.max():.2f}] mm")
print(f"GP phase 2σ max     : {2*gp_std_phase.max():.2f} mm")

# ── Month of minimum PWV (from seasonal GP) ───────────────────────────────────
idx_min       = np.argmin(gp_mean_phase)
phi_min       = phi_pred_flat[idx_min]
month_min_gp  = 1 + phi_min * 12
print(f"Seasonal GP minimum at ϕ = {phi_min:.3f}  → month {month_min_gp:.1f}  ({MONTH_ABBR[int(month_min_gp)-1]})")

---
## 7. Helpers: date axis & style maps

In [ ]:
def make_date_axis(ax_mjd, mjd_min, mjd_max, n_ticks=8):
    """Add a secondary top x-axis showing calendar dates (YYYY-MM)."""
    ax_date = ax_mjd.twiny()
    ax_date.set_xlim(ax_mjd.get_xlim())
    tick_mjds   = np.linspace(mjd_min, mjd_max, n_ticks)
    tick_labels = [
        Time(m, format='mjd', scale='utc').strftime('%Y-%m')
        for m in tick_mjds
    ]
    ax_date.set_xticks(tick_mjds)
    ax_date.set_xticklabels(tick_labels, rotation=30, ha='left', fontsize=9)
    ax_date.set_xlabel('Date (UTC)', fontsize=10, labelpad=6)
    return ax_date


# ── Filter styles ─────────────────────────────────────────────────────────────
FILTERS_OF_INTEREST = ["empty", "OG550_65mm_1", "FELH0600"]

filter_color  = {
    "empty"        : "#1f77b4",
    "OG550_65mm_1" : "#d62728",
    "FELH0600"     : "#2ca02c",
}
filter_marker = {
    "empty"        : "o",
    "OG550_65mm_1" : "s",
    "FELH0600"     : "^",
}
filter_label  = {
    "empty"        : "empty",
    "OG550_65mm_1" : "OG550",
    "FELH0600"     : "FELH0600",
}

# ── Year colours ──────────────────────────────────────────────────────────────
year_cmap  = mpl.colormaps.get_cmap("tab10").resampled(len(years_present))
year_color = {yr: year_cmap(i) for i, yr in enumerate(years_present)}

# ── Season colours (optional fill by season in phase panel) ──────────────────
# Seasons boundaries in fractional year (Southern Hemisphere)
# DJF (Summer): [0, 2/12] ∪ [11/12, 1]
# MAM (Autumn): [2/12, 5/12]
# JJA (Winter): [5/12, 8/12]
# SON (Spring): [8/12, 11/12]
SEASON_COLORS = {
    "Summer (DJF)": "#FFDDC1",   # warm orange-peach
    "Autumn (MAM)": "#C8E6C9",   # soft green
    "Winter (JJA)": "#BBDEFB",   # cool blue
    "Spring (SON)": "#F8BBD0",   # soft pink
}

# ── Month tick positions ──────────────────────────────────────────────────────
month_ticks = [(m - 0.5) / 12.0 for m in range(1, 13)]

print("Filter colours:", filter_color)
print("Year colours:  ", {yr: mpl.colors.to_hex(c) for yr, c in year_color.items()})

---
## 8. Main figure — 2-panel GridSpec

- **Top** : PWV vs MJD — full timeline (2022-2026), GP mean + ±2σ band, date axis
- **Bottom** : year-folded Jan→Dec — GP smooth curve + ±2σ band (no monthly diamonds)

In [ ]:
fig = plt.figure(figsize=(16, 11))
gs  = GridSpec(2, 1, figure=fig, height_ratios=[1.4, 1.0], hspace=0.48)
ax_top = fig.add_subplot(gs[0])
ax_bot = fig.add_subplot(gs[1])

mjd_pred_flat = mjd_pred_full.ravel()

# ==========================================================================
# TOP PANEL — full timeline with GP
# ==========================================================================

# ── individual observations (colour = filter) ─────────────────────────────────
for filt in FILTERS_OF_INTEREST:
    sub = df_keep[df_keep["FILTER"] == filt]
    if len(sub) == 0:
        continue
    ax_top.errorbar(
        sub["MJD"], sub["PWV"], yerr=sub["PWV_err"],
        fmt=filter_marker[filt], color=filter_color[filt],
        markersize=3, linewidth=0, elinewidth=0.6, capsize=1.5,
        alpha=0.45, label=filter_label[filt], zorder=3
    )

# ── GP ±2σ shaded band ────────────────────────────────────────────────────────
ax_top.fill_between(
    mjd_pred_flat,
    np.clip(gp_mean_full - 2*gp_std_full, 0, None),
    gp_mean_full + 2*gp_std_full,
    alpha=0.22, color="#9467bd", label=r"GP $\pm2\sigma$", zorder=4
)

# ── GP mean curve ────────────────────────────────────────────────────────────
ax_top.plot(
    mjd_pred_flat, gp_mean_full,
    color="#9467bd", lw=2.2, zorder=5,
    label="GP mean"
)

ax_top.set_xlabel("MJD", fontsize=12)
ax_top.set_ylabel("PWV [mm]", fontsize=12)
ax_top.set_title(
    "Precipitable Water Vapour at Cerro Pachon — AuxTel/Spectractor (GP regression)",
    fontsize=12, pad=28)
ax_top.set_xlim(mjd_min - 30, mjd_max + 30)
ax_top.set_ylim(PWVMIN, PWVMAX)
ax_top.legend(loc="upper right", fontsize=9, framealpha=0.8)
make_date_axis(ax_top, mjd_min, mjd_max, n_ticks=10)

# ==========================================================================
# BOTTOM PANEL — year-folded with seasonal GP
# ==========================================================================

# ── (Optional) season background shading ─────────────────────────────────────
# Southern Hemisphere seasons in fractional year
# Comment out the block below to remove season colouring
FLAG_SEASON_FILL = False   # set True to enable seasonal background colours
if FLAG_SEASON_FILL:
    season_spans = [
        ("Summer (DJF)", 0.0,     2/12.),
        ("Autumn (MAM)", 2/12.,   5/12.),
        ("Winter (JJA)", 5/12.,   8/12.),
        ("Spring (SON)", 8/12.,  11/12.),
        ("Summer (DJF)", 11/12.,  1.0  ),
    ]
    for (season_name, x0, x1) in season_spans:
        ax_bot.axvspan(x0, x1, alpha=0.18,
                       color=SEASON_COLORS[season_name], zorder=0)

# ── individual observations (colour = year) ───────────────────────────────────
for yr in years_present:
    sub_yr = df_keep[df_keep["year"] == yr]
    for filt in FILTERS_OF_INTEREST:
        sub = sub_yr[sub_yr["FILTER"] == filt]
        if len(sub) == 0:
            continue
        ax_bot.errorbar(
            sub["doy_norm"], sub["PWV"], yerr=sub["PWV_err"],
            fmt=filter_marker[filt], color=year_color[yr],
            markersize=3, linewidth=0, elinewidth=0.5, capsize=1.2,
            alpha=0.45, zorder=3
        )

# ── GP ±2σ shaded band ────────────────────────────────────────────────────────
ax_bot.fill_between(
    phi_pred_flat,
    np.clip(gp_mean_phase - 2*gp_std_phase, 0, None),
    gp_mean_phase + 2*gp_std_phase,
    alpha=0.25, color="#9467bd", label=r"GP $\pm2\sigma$", zorder=4
)

# ── GP mean curve ────────────────────────────────────────────────────────────
ax_bot.plot(
    phi_pred_flat, gp_mean_phase,
    color="#9467bd", lw=2.2, zorder=5,
    label="GP mean (seasonal)"
)

ax_bot.set_xticks(month_ticks)
ax_bot.set_xticklabels(MONTH_ABBR, fontsize=10)
ax_bot.set_xlim(0, 1)
ax_bot.set_ylim(PWVMIN, PWVMAX)
ax_bot.set_xlabel("Month of year", fontsize=12)
ax_bot.set_ylabel("PWV [mm]", fontsize=12)
ax_bot.set_title("Seasonal phase — all years superimposed (GP smoothing)", fontsize=12)

# ── Legend: years + filters + GP ─────────────────────────────────────────────
year_hdl = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=year_color[yr],
           markersize=8, label=str(yr))
    for yr in years_present
]
filt_hdl = [
    Line2D([0],[0], marker=filter_marker[f], color='gray',
           markerfacecolor='gray', markersize=7, linestyle='', label=filter_label[f])
    for f in FILTERS_OF_INTEREST if f in df_keep["FILTER"].values
]
extra_hdl = [
    Line2D([0],[0], color='#9467bd', lw=2.2, label='GP mean'),
    mpatches.Patch(facecolor='#9467bd', alpha=0.3, label=r'GP $\pm2\sigma$'),
]
ax_bot.legend(
    handles=year_hdl + filt_hdl + extra_hdl,
    loc="upper right", fontsize=8.5,
    ncol=max(1, (len(years_present) + 4) // 4),
    framealpha=0.85
)

# ── Figure-level annotation ───────────────────────────────────────────────────
gp_info_text = (
    f"GP (full timeline): RBF×ExpSineSquared(p=365.25 d) + WhiteKernel — "
    f"GP (phase): RBF + WhiteKernel   |   "
    f"Seasonal minimum ≈ {MONTH_ABBR[int(month_min_gp)-1]}"
)
fig.text(0.5, 0.005, gp_info_text, ha="center", fontsize=9,
         bbox=dict(boxstyle="round,pad=0.3", fc="#eef0ff", ec="#9467bd", lw=0.8))
fig.suptitle(f"AuxTel PWV seasonal variations — {version_run} — GP regression",
             fontsize=13, y=1.005)

figfile1 = f"{pathfigs}/{prefix}_{version_run}_2panel_seasonal{figtype}"
fig.savefig(figfile1, bbox_inches="tight")
print(f"Saved: {figfile1}")
plt.show()

---
## 9. Compact single-panel with seasonal GP inset

For a one-column journal figure: main plot = full timeline + GP fit;
inset (upper left) = year-folded seasonal GP.

In [ ]:
fig2, ax_main = plt.subplots(figsize=(14, 6))

for filt in FILTERS_OF_INTEREST:
    sub = df_keep[df_keep["FILTER"] == filt]
    if len(sub) == 0:
        continue
    ax_main.errorbar(
        sub["MJD"], sub["PWV"], yerr=sub["PWV_err"],
        fmt=filter_marker[filt], color=filter_color[filt],
        markersize=3, linewidth=0, elinewidth=0.6, capsize=1.5,
        alpha=0.45, label=filter_label[filt], zorder=3
    )

ax_main.fill_between(
    mjd_pred_flat,
    np.clip(gp_mean_full - 2*gp_std_full, 0, None),
    gp_mean_full + 2*gp_std_full,
    alpha=0.20, color="#9467bd", zorder=4
)
ax_main.plot(mjd_pred_flat, gp_mean_full,
             color="#9467bd", lw=2.2, zorder=5,
             label="GP mean")

ax_main.set_xlabel("MJD", fontsize=12)
ax_main.set_ylabel("PWV [mm]", fontsize=12)
ax_main.set_title(
    "Precipitable Water Vapour — AuxTel/Spectractor, Cerro Pachon (GP)",
    fontsize=12, pad=26)
ax_main.set_xlim(mjd_min - 30, mjd_max + 30)
ax_main.set_ylim(PWVMIN, PWVMAX)
ax_main.legend(loc="upper right", fontsize=9, framealpha=0.85)
make_date_axis(ax_main, mjd_min, mjd_max, n_ticks=10)

# ── inset: seasonal GP ───────────────────────────────────────────────────────
ax_ins = ax_main.inset_axes([0.01, 0.55, 0.33, 0.41])

for yr in years_present:
    sub_yr = df_keep[df_keep["year"] == yr]
    for filt in FILTERS_OF_INTEREST:
        sub = sub_yr[sub_yr["FILTER"] == filt]
        if len(sub) == 0:
            continue
        ax_ins.errorbar(
            sub["doy_norm"], sub["PWV"], yerr=sub["PWV_err"],
            fmt=filter_marker[filt], color=year_color[yr],
            markersize=1, linewidth=0, elinewidth=0.35, capsize=0,
            alpha=0.40, zorder=3
        )

ax_ins.fill_between(
    phi_pred_flat,
    np.clip(gp_mean_phase - 2*gp_std_phase, 0, None),
    gp_mean_phase + 2*gp_std_phase,
    alpha=0.25, color="#9467bd", zorder=4
)
ax_ins.plot(phi_pred_flat, gp_mean_phase,
            color="#9467bd", lw=1.6, zorder=5)

ax_ins.set_xticks(month_ticks[::2])
ax_ins.set_xticklabels(MONTH_ABBR[::2], fontsize=7)
ax_ins.set_xlim(0, 1)
ax_ins.set_ylim(PWVMIN, PWVMAX)
ax_ins.tick_params(axis='y', labelsize=7)
ax_ins.set_ylabel("PWV [mm]", fontsize=8)
ax_ins.set_title("Seasonal GP (all years)", fontsize=8)
ax_ins.grid(True, alpha=0.3)

ins_hdl = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=year_color[yr],
           markersize=5, label=str(yr))
    for yr in years_present
]
ax_ins.legend(handles=ins_hdl, fontsize=6, loc="upper right",
              ncol=2, framealpha=0.7, handlelength=0.8)

figfile2 = f"{pathfigs}/{prefix}_{version_run}_1panel_with_inset{figtype}"
fig2.savefig(figfile2, bbox_inches="tight")
print(f"Saved: {figfile2}")
plt.show()

---
## 10. GP hyperparameters summary

In [ ]:
print("=" * 60)
print("GP FULL TIMELINE — optimised hyperparameters")
print("=" * 60)
print(f"  Kernel : {gp_full.kernel_}")
print(f"  Log-MLL: {gp_full.log_marginal_likelihood_value_:.3f}")
print()
print("=" * 60)
print("GP SEASONAL PHASE — optimised hyperparameters")
print("=" * 60)
print(f"  Kernel : {gp_phase.kernel_}")
print(f"  Log-MLL: {gp_phase.log_marginal_likelihood_value_:.3f}")
print()
print(f"Seasonal minimum (GP) ~ month {month_min_gp:.1f}  ({MONTH_ABBR[int(month_min_gp)-1]})")

---
## 11. Summary & JSON export

In [ ]:
gp_results = {
    "version_run"                    : version_run,
    "n_daily_points"                 : int(len(df_daily)),
    "n_raw_points"                   : int(len(df_keep)),
    "gp_full_kernel"                 : str(gp_full.kernel_),
    "gp_full_log_mll"               : float(gp_full.log_marginal_likelihood_value_),
    "gp_phase_kernel"                : str(gp_phase.kernel_),
    "gp_phase_log_mll"              : float(gp_phase.log_marginal_likelihood_value_),
    "gp_mean_full_min_mm"           : float(gp_mean_full.min()),
    "gp_mean_full_max_mm"           : float(gp_mean_full.max()),
    "gp_mean_phase_min_mm"          : float(gp_mean_phase.min()),
    "gp_mean_phase_max_mm"          : float(gp_mean_phase.max()),
    "seasonal_minimum_month_gp"     : float(month_min_gp),
    "seasonal_minimum_month_name_gp": MONTH_ABBR[int(month_min_gp)-1],
}

outjson = f"{pathfigs}/pwv_gp_fit_{version_run}.json"
with open(outjson, "w") as fh:
    json.dump(gp_results, fh, indent=2)
print(f"Saved: {outjson}")
print(json.dumps(gp_results, indent=2))

---
## 12. Seasonal PWV statistics

Seasons defined for the **Southern Hemisphere** (Cerro Pachon):

| Season | Months |
|--------|--------|
| Summer (DJF) | December, January, February |
| Autumn (MAM) | March, April, May |
| Winter (JJA) | June, July, August |
| Spring (SON) | September, October, November |

Statistics computed over **all years** and **all PWV-sensitive filters**
after quality cuts:
- **N** : number of individual measurements
- **Mean** : arithmetic mean [mm]
- **Median** : median [mm]
- **Std** : standard deviation [mm]
- **GP mean** : GP posterior mean averaged over season [mm]
- **GP 2σ** : mean ±2σ width over season [mm]

In [ ]:
# ── Season definition — Southern Hemisphere ───────────────────────────────────
SEASON_MAP = {
    12: "Summer (DJF)",  1: "Summer (DJF)",  2: "Summer (DJF)",
     3: "Autumn (MAM)",  4: "Autumn (MAM)",  5: "Autumn (MAM)",
     6: "Winter (JJA)",  7: "Winter (JJA)",  8: "Winter (JJA)",
     9: "Spring (SON)", 10: "Spring (SON)", 11: "Spring (SON)",
}
SEASON_ORDER = ["Summer (DJF)", "Autumn (MAM)", "Winter (JJA)", "Spring (SON)"]

# Season boundaries in fractional year (doy_norm)
SEASON_BOUNDS = {
    "Summer (DJF)": [(0.0, 2/12.), (11/12., 1.0)],
    "Autumn (MAM)": [(2/12., 5/12.)],
    "Winter (JJA)": [(5/12., 8/12.)],
    "Spring (SON)": [(8/12., 11/12.)],
}

df_keep["season"] = df_keep["month"].map(SEASON_MAP)


def gp_season_mean(bounds_list, phi_arr, mean_arr, std_arr):
    """Average GP posterior mean and mean 2σ over a list of (phi_lo, phi_hi) segments."""
    masks = np.zeros(len(phi_arr), dtype=bool)
    for (lo, hi) in bounds_list:
        masks |= (phi_arr >= lo) & (phi_arr < hi)
    return mean_arr[masks].mean(), (2*std_arr[masks]).mean()


rows = []
for season in SEASON_ORDER:
    sub  = df_keep[df_keep["season"] == season]
    vals = sub["PWV"].values
    errs = sub["PWV_err"].values
    mask = np.isfinite(vals) & np.isfinite(errs) & (errs > 0)
    vals, errs = vals[mask], errs[mask]

    gp_mu, gp_2s = gp_season_mean(
        SEASON_BOUNDS[season], phi_pred_flat, gp_mean_phase, gp_std_phase
    )

    if len(vals) == 0:
        rows.append({"Season": season, "N": 0,
                     "Mean [mm]": np.nan, "Median [mm]": np.nan,
                     "Std [mm]": np.nan,
                     "GP mean [mm]": gp_mu, "GP 2σ [mm]": gp_2s})
        continue

    rows.append({
        "Season"      : season,
        "N"           : len(vals),
        "Mean [mm]"   : np.mean(vals),
        "Median [mm]" : np.median(vals),
        "Std [mm]"    : np.std(vals, ddof=1),
        "GP mean [mm]": gp_mu,
        "GP 2σ [mm]"  : gp_2s,
    })

df_seasons = pd.DataFrame(rows).set_index("Season")

# ── styled display ────────────────────────────────────────────────────────────
fmt_dict = {c: "{:.3f}" for c in df_seasons.columns if c != "N"}
fmt_dict["N"] = "{:d}"

display(
    df_seasons.style
    .format(fmt_dict)
    .set_caption(
        f"Seasonal PWV statistics — {version_run} (GP smoothing, Southern Hemisphere)"
    )
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-weight", "bold"), ("font-size", "13px"),
                   ("text-align", "left"), ("padding-bottom", "8px")]},
        {"selector": "th",
         "props": [("background-color", "#e8d5f5"), ("font-weight", "bold"),
                   ("text-align", "center")]},
        {"selector": "td",
         "props": [("text-align", "right"), ("padding", "4px 10px")]},
        {"selector": "tr:nth-child(even)",
         "props": [("background-color", "#f8f5fc")]},
        {"selector": "tr:hover",
         "props": [("background-color", "#f3e5ff")]},
    ])
    .bar(subset=["Mean [mm]", "Median [mm]", "GP mean [mm]"],
         color="#c9a8f0", vmin=0)
)

In [ ]:
# ── LaTeX table ───────────────────────────────────────────────────────────────
header_latex = {
    "N"           : r"$N$",
    "Mean [mm]"   : r"$\langle\mathrm{PWV}\rangle$ [mm]",
    "Median [mm]" : r"$\widetilde{\mathrm{PWV}}$ [mm]",
    "Std [mm]"    : r"$\sigma$ [mm]",
    "GP mean [mm]": r"$\mu_{\rm GP}$ [mm]",
    "GP 2σ [mm]"  : r"$2\sigma_{\rm GP}$ [mm]",
}

col_spec = "l" + "r" * len(df_seasons.columns)

lines = []
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(
    r"\caption{Seasonal statistics of the precipitable water vapour (PWV) "
    r"at Cerro Pach\'{o}n measured by AuxTel/Spectractor after quality cuts. "
    r"Seasons are defined for the Southern Hemisphere (Summer = DJF, "
    r"Autumn = MAM, Winter = JJA, Spring = SON). "
    r"$\langle\mathrm{PWV}\rangle$: arithmetic mean; "
    r"$\widetilde{\mathrm{PWV}}$: median; "
    r"$\sigma$: standard deviation; "
    r"$\mu_{\rm GP}$ and $2\sigma_{\rm GP}$: Gaussian Process posterior "
    r"mean and $\pm2\sigma$ width averaged over each season.}"
)
lines.append(r"\label{tab:pwv_seasonal_gp}")
lines.append(rf"\begin{{tabular}}{{{col_spec}}}")
lines.append(r"\hline\hline")

header_cells = ["Season"] + [header_latex.get(c, c) for c in df_seasons.columns]
lines.append(" & ".join(header_cells) + r" \\")
lines.append(r"\hline")

for season, row in df_seasons.iterrows():
    cells = [season]
    for col in df_seasons.columns:
        if col == "N":
            cells.append(str(int(row[col])))
        else:
            cells.append(f"{row[col]:.3f}")
    lines.append(" & ".join(cells) + r" \\")

lines.append(r"\hline")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

latex_str = "\n".join(lines)
print(latex_str)

latex_file = f"{pathfigs}/{prefix}_{version_run}_seasonal_stats_table.tex"
with open(latex_file, "w") as fh:
    fh.write(latex_str + "\n")
print(f"\nSaved LaTeX table: {latex_file}")